# 🧠 Stage 2 — Memory Agent
**CodePilot AI Studio | Module 4 | Agentic AI in Software Engineering**

---

## What You Will Learn
- Why an LLM **forgets** everything — and why that is a problem
- What **short-term memory** means for an AI agent
- How `ConversationBufferMemory` stores the full chat history
- How `PromptTemplate` injects memory into every LLM call automatically
- How `ConversationChain` wires LLM + memory + prompt together

## The Problem We Are Solving
```
WITHOUT memory:                      WITH memory:
Turn 1: 'What is IndexError?'        Turn 1: 'What is IndexError?'
Agent : 'Index out of range...'      Agent : 'Index out of range...'
Turn 2: 'Show me an example'         Turn 2: 'Show me an example'
Agent : 'Example of WHAT?' WRONG!    Agent : 'Here is IndexError...' CORRECT!
```

> **Analogy:** A doctor who reads your full medical history before every appointment.

⏱ **Expected time: 20 minutes**

## Step 1 — Paste Your Groq API Key
**Get your free key at https://console.groq.com → API Keys → Create API Key**

Steps:
1. Go to **https://console.groq.com**
2. Sign up free (Google account or email — no credit card)
3. Click **API Keys** in the left sidebar
4. Click **Create API Key** → name it anything → click Submit
5. **Copy the key** (looks like `gsk_xxxx...`) — save it in Notepad
6. Paste it below between the quotes

> Each student must use their own key. Never share your key.

In [ ]:
# ── GROQ API KEY ─────────────────────────────────────────────
# Paste your key between the quotes below.
# Get it free from: https://console.groq.com → API Keys → Create API Key

GROQ_API_KEY = "paste-your-groq-key-here"   # ← replace this!

# Set as environment variable so LangChain can find it automatically
import os
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Quick check
if GROQ_API_KEY == "paste-your-groq-key-here" or len(GROQ_API_KEY) < 20:
    print("ERROR: Please paste your real Groq API key above!")
    print("Get it from: https://console.groq.com → API Keys")
else:
    print(f"API key accepted! Starts with: {GROQ_API_KEY[:8]}...")
    print("Ready to proceed to the next cell.")

## Step 2 — Setup

In [ ]:
print("Installing packages...")
!pip install -q langchain-groq langchain langchain-community
print("Setup complete!")

## Step 3 — Understand Memory Before Using It

### How ConversationBufferMemory works
```
Every time chain.predict() is called:

 New message ──────────────────────────────────────┐
                                                   ↓
 Memory loads history ──► PromptTemplate ──► LLM ──► Response
  fills {chat_history}     fills {input}              │
                                                       ↓
                                            Memory saves Q+A
```
The key point: the agent re-reads ALL history before every single reply.
That is how it 'remembers' — it does not actually store memories like a human.
It just re-reads the conversation transcript every time!

### The PromptTemplate
```
template = """
You are CodePilot...
Conversation history:
{chat_history}    ← filled automatically by memory
Student: {input}  ← filled with new message
CodePilot:"""
```

In [ ]:
# ── CONCEPT DEMO: Look inside memory ─────────────────────────
# This cell shows what memory stores — NO LLM call here.
# Just demonstrating the memory object so you understand it.

from langchain.memory import ConversationBufferMemory

# Create memory
# memory_key='chat_history' MUST match {chat_history} in the prompt template
# return_messages=True stores as structured objects (not plain text)
demo_memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Simulate two turns of conversation by adding messages manually
demo_memory.chat_memory.add_user_message("What is IndexError?")
demo_memory.chat_memory.add_ai_message("IndexError: list index out of range.")
demo_memory.chat_memory.add_user_message("Show me a code example.")
demo_memory.chat_memory.add_ai_message("my_list=[1,2,3]; print(my_list[5]) → IndexError")

# Print what is stored — this is what the agent reads before every reply!
print("What memory stores after 2 turns (what agent reads before every reply):")
print("-" * 65)
for i, msg in enumerate(demo_memory.chat_memory.messages):
    sender = "Student  " if "Human" in type(msg).__name__ else "CodePilot"
    print(f"[{i+1}] {sender}: {msg.content}")
print()
print(f"Total messages: {len(demo_memory.chat_memory.messages)}")
print()
print("KEY INSIGHT: The agent re-reads the ENTIRE list before every reply.")
print("That is the whole secret of short-term memory!")

## Step 4 — The Full Memory Agent
Now let us build the real agent with live LLM calls.
Watch how Turn 2 and Turn 3 **reference earlier turns** — proof memory works!

In [ ]:
# ============================================================
# CodePilot AI Studio — Stage 2: Memory Agent
# ============================================================
# Concept  : Short-term memory using ConversationBufferMemory
# New here : ConversationBufferMemory, PromptTemplate, ConversationChain
# ============================================================

from langchain_groq import ChatGroq
from langchain.memory import ConversationBufferMemory  # stores chat history
from langchain.chains import ConversationChain         # wires everything together
from langchain.prompts import PromptTemplate           # fills in history + message

# ── STEP 1: Connect to LLM ────────────────────────────────────
# temperature=0.7 → good balance for conversational responses
llm = ChatGroq(model="llama3-8b-8192", temperature=0.7)

# ── STEP 2: Create memory object ──────────────────────────────
# Starts empty. Grows as the conversation continues.
# memory_key MUST match the variable name in the prompt template below
memory = ConversationBufferMemory(
    memory_key="chat_history",   # must match {chat_history} in template
    return_messages=True          # store as objects, not raw text
)

# ── STEP 3: Define the prompt template ────────────────────────
# {chat_history} → auto-filled from memory before every call
# {input}        → the student's current message
# LangChain replaces these placeholders automatically
prompt_template = PromptTemplate(
    input_variables=["chat_history", "input"],
    template="""You are CodePilot, a helpful Python coding assistant.
You remember everything from our conversation. Be concise.

Conversation so far:
{chat_history}

Student: {input}
CodePilot:"""
)

# ── STEP 4: Create the ConversationChain ──────────────────────
# ConversationChain wires together:
#   llm      → the brain (Llama 3)
#   memory   → the notepad (ConversationBufferMemory)
#   prompt   → the template (fills in history + new message)
# After every predict() call, it automatically saves Q+A to memory
chain = ConversationChain(
    llm=llm,
    memory=memory,
    prompt=prompt_template,
    verbose=False    # change to True to see the full prompt each time!
)

# ── STEP 5: Run a 3-turn conversation ─────────────────────────
print("=" * 60)
print("CodePilot AI Studio — Stage 2: Memory Agent")
print("=" * 60)
print()

# Turn 1: First question — memory is empty, agent answers fresh
q1 = "What is an IndexError in Python?"
print(f"Student (Turn 1): {q1}")
r1 = chain.predict(input=q1)
print(f"CodePilot: {r1}")
print()

# Turn 2: 'that error' only makes sense with Turn 1 in memory!
q2 = "Can you give me a code example of that error?"
print(f"Student (Turn 2): {q2}")
r2 = chain.predict(input=q2)   # memory now has Turn 1
print(f"CodePilot: {r2}")
print()

# Turn 3: 'the bug you showed me' needs Turns 1 AND 2!
q3 = "How do I fix the bug you just showed me?"
print(f"Student (Turn 3): {q3}")
r3 = chain.predict(input=q3)   # memory has Turns 1 + 2
print(f"CodePilot: {r3}")
print()

# Show what is in memory after the conversation
print("=" * 60)
print("Memory contents after 3 turns:")
print("=" * 60)
for i, msg in enumerate(memory.chat_memory.messages):
    sender = "Student  " if "Human" in type(msg).__name__ else "CodePilot"
    print(f"[{i+1}] {sender}: {msg.content[:70]}...")
print()
print(f"Total messages in memory: {len(memory.chat_memory.messages)}")
print()
print("SHORT-TERM MEMORY: lives only while this notebook session runs.")
print("Restart the session = memory gone. Stage 4 (RAG) fixes that!")

## Step 5 — Verify

In [ ]:
try:
    count = len(memory.chat_memory.messages)
    if count >= 6:
        print("VERIFICATION PASSED")
        print(f"  Turns completed    : {count // 2}")
        print(f"  Messages in memory : {count}")
        print("Ready for Stage 3!")
    else:
        print(f"Only {count} messages found. Run Step 4 first.")
except:
    print("Run Step 4 first.")

## Step 6 — Experiments

In [ ]:
# ── EXPERIMENT 1: Prove memory works ──────────────────────────
# Ask the agent what the very first question was.
# It should remember because it re-reads all history!

result = chain.predict(input="What was the very first question I asked you?")
print("Test: Can agent remember Turn 1?")
print(f"CodePilot: {result}")
print()
print("It should mention IndexError — that was Turn 1!")

In [ ]:
# ── EXPERIMENT 2: Turn on verbose mode ────────────────────────
# This shows you the FULL prompt sent to Llama 3.
# You will see chat_history filled in with all previous messages.
# This is the most educational experiment in Stage 2!

from langchain.memory import ConversationBufferMemory
verbose_chain = ConversationChain(
    llm=llm,
    memory=ConversationBufferMemory(memory_key="chat_history", return_messages=True),
    prompt=prompt_template,
    verbose=True   # ← LOOK at the full prompt printed below!
)
print("=== Turn 1 (verbose mode ON) ===")
verbose_chain.predict(input="What is a for loop?")
print()
print("=== Turn 2 — notice chat_history is now filled! ===")
verbose_chain.predict(input="Show me an example of that.")

In [ ]:
# ── EXPERIMENT 3: Clear memory and test ───────────────────────
# What happens when we wipe memory mid-conversation?

print(f"Before clearing: {len(memory.chat_memory.messages)} messages")
memory.clear()
print(f"After clearing : {len(memory.chat_memory.messages)} messages")
print()

result = chain.predict(input="How do I fix the bug you showed me?")
print(f"Agent response (after memory clear):")
print(result)
print()
print("Notice: the agent says 'which bug?' — it forgot everything!")
print("This PROVES that memory was responsible for remembering before.")

## Summary — Stage 2 Complete!

| Concept | What it means |
|---------|---------------|
| **Stateless LLM** | Forgets after every single call |
| **Short-term memory** | Remembers within one session |
| **`ConversationBufferMemory`** | Stores all messages as a list |
| **`PromptTemplate`** | Fills in {chat_history} and {input} |
| **`ConversationChain`** | Glue connecting LLM + memory + prompt |
| **`chain.predict()`** | Send message → get reply → save to memory |

---
### Problem with Stage 2
One generic prompt handles everything — debug, explain, review — all get the same treatment.

**Stage 3 fixes this with specialised skill prompts and intent detection.**

➡️ Open `stage3_tool_agent/Stage3_Tool_Agent.ipynb`